# 章节实践

请运用本章所学的调试打印方法，对以下提供的sinh算子程序完整源代码进行调试。通过分析打印信息和日志，定位并解决代码中的问题，使最终运行结果符合预期。

---

## 环境准备

In [1]:
!mkdir -p Sources/07.05

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))

current_path = !pwd
current_path = current_path[0]
log_abs_path = f"{current_path}/Sources/07.05/log/"
os.makedirs(log_abs_path, exist_ok=True)
os.environ['ASCEND_PROCESS_LOG_PATH'] = log_abs_path
print("\n🎉 Environment initialization process completed successfully!")

## 调试任务

以下源码中存在三处错误，请运用本章所学的调试方法（如添加 printf 打印、查看 Plog 日志等）定位问题，并修改源码，直至程序运行结果正确。

In [ ]:
%%writefile Sources/07.05/sinh_custom.asc

#include <cstdio>
#include <cstdlib>
#include <cstdint>
#include <iostream>
#include <vector>
#include <algorithm>
#include <iterator>
#include <cmath>
#include <ctime>
#include <random>
#include "acl/acl.h"
#include "kernel_operator.h"

#define CHECK_ACL(expr) \
    do { \
        auto __ret = (expr); \
        int32_t __code = static_cast<int32_t>(__ret); \
        if (__code != 0) { \
            fprintf(stderr, "[ERROR] %s failed at %s:%d, ret=%d\n", #expr, __FILE__, __LINE__, __code); \
            exit(-1); \
        } else { \
            fprintf(stdout, "[INFO] %s succeeded at %s:%d\n", #expr, __FILE__, __LINE__); \
        } \
    } while (0)
                    
constexpr uint32_t BUFFER_NUM = 2; // tensor num for each queue

struct SinhCustomTilingData
{
    uint32_t totalLength;
    uint32_t tileNum;
};

class KernelSinh {
public:
    __aicore__ inline KernelSinh() {}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
    {
        this->blockLength = totalLength / AscendC::GetBlockNum();
        this->tileNum = tileNum;
        this->tileLength = this->blockLength / tileNum / BUFFER_NUM;
        xGm.SetGlobalBuffer((__gm__ float *)x + this->blockLength, this->blockLength);
        zGm.SetGlobalBuffer((__gm__ float *)z + this->blockLength, this->blockLength);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileLength * sizeof(float));
        pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileLength * sizeof(float));
    }
    __aicore__ inline void Process()
    {
        int32_t loopCount = this->tileNum * BUFFER_NUM;
        for (int32_t i = 0; i < loopCount; i++) {
            CopyIn(i);
            Compute(i);
            CopyOut(i);
        }
    }

private:
    __aicore__ inline void CopyIn(int32_t progress)
    {
        AscendC::LocalTensor<float> xLocal = inQueueX.AllocTensor<float>();
        AscendC::DataCopy(xLocal, xGm[this->tileLength], this->tileLength);
        inQueueX.EnQue(xLocal);
    }
    __aicore__ inline void Compute(int32_t progress)
    {
        AscendC::LocalTensor<float> xLocal = inQueueX.DeQue<float>();
        AscendC::LocalTensor<float> zLocal = outQueueZ.AllocTensor<float>();
        Muls(zLocal, xLocal, (float)-1.0, this->tileLength);
        Exp(zLocal, zLocal, this->tileLength);
        Exp(xLocal, xLocal, this->tileLength);
        Sub(zLocal, xLocal, zLocal, this->tileLength);
        Muls(zLocal, zLocal, (float)0.5, this->tileLength);
        outQueueZ.EnQue<float>(zLocal);
        inQueueX.FreeTensor(xLocal); // 仅释放xLocal
    }
    __aicore__ inline void CopyOut(int32_t progress)
    {
        AscendC::LocalTensor<float> zLocal = outQueueZ.DeQue<float>();
        AscendC::DataCopy(zGm[progress * this->tileLength], zLocal, this->tileLength);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX;
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<float> xGm;
    AscendC::GlobalTensor<float> zGm;
    uint32_t blockLength;
    uint32_t tileNum;
    uint32_t tileLength;
};

__global__ __aicore__ void sinh_custom(GM_ADDR x, GM_ADDR z, SinhCustomTilingData tiling)
{
    KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
    KernelSinh op;
    op.Init(x, z, tiling.totalLength, tiling.tileNum); // 仅传入x和z
    op.Process();
}

std::vector<float> kernel_sinh(std::vector<float> &x)
{
    constexpr uint32_t blockDim = 8;
    uint32_t totalLength = x.size();
    size_t totalByteSize = totalLength * sizeof(float);
    int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    SinhCustomTilingData tiling = {/*totalLength:*/totalLength, /*tileNum:*/8};
    uint8_t *xHost = reinterpret_cast<uint8_t *>(x.data());
    uint8_t *zHost = nullptr;
    uint8_t *xDevice = nullptr;
    uint8_t *zDevice = nullptr;

    CHECK_ACL(aclInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));
    CHECK_ACL(aclrtCreateStream(&stream));

    CHECK_ACL(aclrtMallocHost((void **)(&zHost), totalByteSize));
    CHECK_ACL(aclrtMalloc((void **)&xDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc((void **)&zDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMemcpy(xDevice, totalByteSize, xHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE));

    sinh_custom<<<blockDim, nullptr, stream>>>(xDevice, zDevice, tiling);
    CHECK_ACL(aclrtSynchronizeStream(stream));

    CHECK_ACL(aclrtMemcpy(zHost, totalByteSize, zDevice, totalByteSize, ACL_MEMCPY_DEVICE_TO_HOST));
    std::vector<float> z((float *)zHost, (float *)(zHost + totalByteSize));

    // 释放资源也可以检查（可选）
    CHECK_ACL(aclrtFree(xDevice));
    CHECK_ACL(aclrtFree(zDevice));
    CHECK_ACL(aclrtFreeHost(zHost));

    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclFinalize());
    return z;
}

uint32_t VerifyResult(std::vector<float> &output, std::vector<float> &golden)
{
    auto printTensor = [](std::vector<float> &tensor, const char *name) {
        constexpr size_t maxPrintSize = 20;
        std::cout << name << ": ";
        std::copy(tensor.begin(), tensor.begin() + std::min(tensor.size(), maxPrintSize),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > maxPrintSize) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };
    printTensor(output, "Output");
    printTensor(golden, "Golden");
    bool pass = true;
    constexpr float eps = 1e-3;
    for (size_t i = 0; i < golden.size(); i++) {
        if (fabs(output[i] - golden[i]) > eps) {
            pass = false;
            break;
        }
    }
    if (pass) {
        std::cout << "[Success] Case accuracy is verification passed." << std::endl;
        return 0;
    } else {
        std::cout << "[Failed] Case accuracy is verification failed!" << std::endl;
        return 1;
    }
}


int32_t main(int32_t argc, char *argv[])
{
    constexpr uint32_t totalLength = 8 * 2048;
    srand(time(0));
    std::vector<float> x(totalLength);
    constexpr float rand_max_float = static_cast<float>(RAND_MAX);
    for (auto& val : x) {
        val = (static_cast<float>(rand()) / rand_max_float) * 10.0f;
    }
    std::vector<float> output = kernel_sinh(x);
    std::vector<float> golden;
    golden.reserve(totalLength);
    for (const auto& val : x) {
        golden.push_back(sinh(val));
    }
    return VerifyResult(output, golden);
}

## 编译运行

执行以下命令编译并运行程序，观察打印输出的调试信息及最终运行结果。

In [ ]:
%%writefile Sources/07.05/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)

find_package(ASC REQUIRED)

project(kernel_samples LANGUAGES ASC CXX)

add_executable(demo
    sinh_custom.asc
)

target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=dav-2201>
)

target_link_libraries(demo PRIVATE m)

In [ ]:
!rm -rf Sources/07.05/log/*
!cd Sources/07.05 && \
mkdir -p build
!export ASC_DIR=$ASCEND_HOME_PATH/aarch64-linux/tikcpp/ascendc_kernel_cmake/ && \
cd Sources/07.05/build/ && \
cmake .. && \
make
!cd Sources/07.05/build/ && \
./demo

## 查看Plog信息

执行以下命令，查看最新的Plog日志信息，辅助定位问题：

In [ ]:
!cd Sources/07.05/log/debug/plog && \
grep -nr "ERROR"

## 预期执行效果

```
[Success] Case accuracy is verification passed.
```

## 验证结果

执行以下代码获取答案。

In [ ]:
!cat ./answer/07.05_answer.txt